# Partie 2 : traitement batch avec SparkCore (RDD)

Fichier d'entrée : `lake/bronze/history/transactions_history.jsonl` (50 000 événements générés par
`producer.py --mode file --n 50000`).

Pendant l'exécution, ouvrir la Spark UI de l'application : <http://localhost:4040> (onglets **Jobs** et **Stages**).

In [1]:
import json, os
from operator import add
import pandas as pd
from sahlpay_spark import get_spark, chrono, LAKE

spark = get_spark("SahlPay-Partie2-RDD")
sc = spark.sparkContext

HISTORY = f"{LAKE}/bronze/history/transactions_history.jsonl"
MERCHANTS = f"{LAKE}/reference/merchants.json"
MERCHANT_ID = "merchant_id"          # nom du champ identifiant dans merchants.json (à adapter si besoin)
OUT = os.environ.get("SAHLPAY_OUT", "../outputs")
os.makedirs(OUT, exist_ok=True)
temps = {}

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 21:14:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 47412)
Traceback (most recent call last):
  File "/usr/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.8/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "

## 1. Chargement en RDD et parsing JSON

`textFile`, `map` et `filter` sont des **transformations** : rien n'est calculé tant qu'on n'appelle pas une
**action** (`count`, `take`, `collect`…).

In [2]:
def parse(line):
    try:
        return json.loads(line)
    except (json.JSONDecodeError, TypeError):
        return None                      # ligne illisible : ignorée

raw = sc.textFile(HISTORY)                                   # transformation
events = raw.map(parse).filter(lambda e: e is not None)      # transformations

with chrono("RDD - chargement + count", temps):
    n_events = events.count()                                # action
print("Événements :", n_events, "| partitions :", events.getNumPartitions())
events.take(2)                                               # action

26/11/25 21:14:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 0:>                                                          (0 + 2) / 2]

[temps] RDD - chargement + count : 4.328 s
Événements : 50000 | partitions : 2


[{'tx_id': 'TX-00000001',
  'type': 'P2M',
  'payer_wallet': 'W-14188',
  'payee_id': 'M-245',
  'device_id': 'D-4dae40',
  'city': 'Casablanca',
  'channel': 'APP',
  'amount_mad': 157.06,
  'status': 'INITIATED',
  'event_time': '2026-11-01T12:57:12Z'},
 {'tx_id': 'TX-00000001',
  'type': 'P2M',
  'payer_wallet': 'W-14188',
  'payee_id': 'M-245',
  'device_id': 'D-4dae40',
  'city': 'Casablanca',
  'channel': 'APP',
  'amount_mad': 157.06,
  'status': 'AUTHORIZED',
  'event_time': '2026-11-01T12:57:13Z'}]

## 2. Filtrer les transactions `COMPLETED`

On met ce RDD en cache car il est réutilisé par toutes les questions suivantes.

In [3]:
completed = events.filter(lambda e: e.get("status") == "COMPLETED").cache()

with chrono("RDD - filtre COMPLETED + count", temps):
    n_completed = completed.count()
print("Transactions COMPLETED :", n_completed)

[temps] RDD - filtre COMPLETED + count : 0.655 s
Transactions COMPLETED : 15798


## 3. Montant total encaissé par ville (`map` + `reduceByKey`)

In [4]:
total_par_ville = (completed
                   .map(lambda e: (e["city"], float(e["amount_mad"])))
                   .reduceByKey(add))

with chrono("RDD - total par ville (reduceByKey)", temps):
    totaux = total_par_ville.collect()

for ville, total in sorted(totaux, key=lambda kv: -kv[1]):
    print(f"{ville:<12} {total:>15,.2f} MAD")

[temps] RDD - total par ville (reduceByKey) : 0.562 s
Casablanca      1,471,886.03 MAD
Rabat             697,455.61 MAD
Marrakech         470,642.44 MAD
Fes               411,967.55 MAD
Tanger            403,067.27 MAD
Agadir            287,146.05 MAD
Meknes            234,613.87 MAD
Kenitra           231,810.18 MAD
Oujda             211,568.72 MAD
Tetouan           204,449.53 MAD


## 4. Montant moyen par transaction et par ville, en un seul passage (`aggregateByKey`)

L'accumulateur `(somme, nombre)` est construit dans chaque partition (`seqOp`), puis les accumulateurs
partiels sont fusionnés après le shuffle (`combOp`). On obtient total, nombre et moyenne sans relire les données.

In [5]:
somme_nb = (completed
            .map(lambda e: (e["city"], float(e["amount_mad"])))
            .aggregateByKey((0.0, 0),
                            lambda acc, x: (acc[0] + x, acc[1] + 1),          # seqOp (dans une partition)
                            lambda a, b: (a[0] + b[0], a[1] + b[1])))         # combOp (entre partitions)
stats_ville = somme_nb.mapValues(lambda s: (round(s[0], 2), s[1], round(s[0] / s[1], 2)))

sc.setJobGroup("agg_ville", "Total et moyenne par ville")
with chrono("RDD - total + moyenne par ville (aggregateByKey)", temps):
    stats = stats_ville.collect()

table_villes = (pd.DataFrame([(v, *s) for v, s in stats],
                             columns=["ville", "montant_total_mad", "nb_transactions", "montant_moyen_mad"])
                  .sort_values("montant_total_mad", ascending=False)
                  .reset_index(drop=True))
table_villes.to_csv(f"{OUT}/partie2_villes.csv", index=False)
table_villes

[temps] RDD - total + moyenne par ville (aggregateByKey) : 0.300 s


,ville,montant_total_mad,nb_transactions,montant_moyen_mad
0,Casablanca,1471886.03,5010,293.79
1,Rabat,697455.61,2695,258.80
2,Marrakech,470642.44,1741,270.33
3,Fes,411967.55,1481,278.17
4,Tanger,403067.27,1418,284.25
5,Agadir,287146.05,1089,263.68
6,Meknes,234613.87,554,423.49
7,Kenitra,231810.18,602,385.07
8,Oujda,211568.72,532,397.69
9,Tetouan,204449.53,676,302.44


## 5. Top 5 des commerçants par nombre de paiements validés

Paiements validés = transactions `P2M` au statut `COMPLETED`. Les noms viennent du référentiel, diffusé
aux Executors avec une variable **broadcast** (petite table de 300 lignes).

In [6]:
noms = (sc.textFile(MERCHANTS)
          .map(json.loads)
          .map(lambda m: (m[MERCHANT_ID], m.get("name", "")))
          .collectAsMap())
noms_b = sc.broadcast(noms)

with chrono("RDD - top 5 commerçants (takeOrdered)", temps):
    top5 = (completed
            .filter(lambda e: e["type"] == "P2M")
            .map(lambda e: (e["payee_id"], 1))
            .reduceByKey(add)
            .takeOrdered(5, key=lambda kv: -kv[1]))           # action

table_top5 = pd.DataFrame([(m, noms_b.value.get(m, "?"), n) for m, n in top5],
                          columns=["merchant_id", "nom", "nb_paiements_valides"])
table_top5.to_csv(f"{OUT}/partie2_top5_commercants.csv", index=False)
table_top5

[temps] RDD - top 5 commerçants (takeOrdered) : 0.328 s


,merchant_id,nom,nb_paiements_valides
0,M-225,Rif Restauration 225,49
1,M-152,Andalous Habillement 152,47
2,M-252,Rif Pharmacie 252,47
3,M-045,Yasmine Epicerie 45,47
4,M-253,Sahara E-commerce 253,45


## 6. Lineage : `toDebugString()`

Chaque niveau d'indentation (`+-`) marque une frontière de **shuffle**, donc un nouveau stage.

In [7]:
lineage = stats_ville.toDebugString().decode("utf-8")
print(lineage)
with open(f"{OUT}/partie2_lineage.txt", "w") as f:
    f.write(lineage)

(2) PythonRDD[15] at collect at /tmp/ipykernel_2108/2545759774.py:10 []
 |  MapPartitionsRDD[14] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[13] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[12] at aggregateByKey at /tmp/ipykernel_2108/2545759774.py:1 []
    |  PythonRDD[11] at aggregateByKey at /tmp/ipykernel_2108/2545759774.py:1 []
    |  PythonRDD[4] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 848.8 KiB; DiskSize: 0.0 B
    |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
    |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


| Opération utilisée | Type | Pourquoi |
|---|---|---|
| `textFile`, `map`, `filter`, `mapValues` | Transformation (étroite) | Chaque partition est traitée seule, sans échange de données |
| `reduceByKey`, `aggregateByKey` | Transformation (large) | Regroupe les clés de toutes les partitions : **shuffle** |
| `cache` | Marqueur | Garde le RDD en mémoire au premier calcul |
| `count`, `take`, `collect`, `collectAsMap`, `takeOrdered` | **Action** | Déclenche réellement le calcul et renvoie un résultat au Driver |

## 7. Stages du job `aggregateByKey` (à comparer avec le DAG de la Spark UI)

In [8]:
tracker = sc.statusTracker()
for job_id in tracker.getJobIdsForGroup("agg_ville"):
    job = tracker.getJobInfo(job_id)
    print(f"Job {job_id} : {len(job.stageIds)} stages -> {list(job.stageIds)}")
    for sid in job.stageIds:
        st = tracker.getStageInfo(sid)
        if st:
            print(f"   stage {sid} : {st.name} | {st.numTasks} tâches")
sc.setJobGroup(None, None)

Job 6 : 2 stages -> [9, 8]
   stage 9 : takeOrdered at /tmp/ipykernel_2108/3530939360.py:8 | 2 tâches
   stage 8 : reduceByKey at /tmp/ipykernel_2108/3530939360.py:8 | 2 tâches
Job 5 : 1 stages -> [7]
   stage 7 : collectAsMap at /tmp/ipykernel_2108/3530939360.py:1 | 2 tâches
Job 4 : 2 stages -> [5, 6]
   stage 5 : aggregateByKey at /tmp/ipykernel_2108/2545759774.py:1 | 2 tâches
   stage 6 : collect at /tmp/ipykernel_2108/2545759774.py:10 | 2 tâches


**Lecture attendue :** 2 stages. Le stage 1 lit le fichier, filtre et fait les `map` puis une agrégation locale
par partition (côté *map* du shuffle). Le stage 2 récupère par le réseau les sommes partielles de chaque ville
(shuffle read) et les fusionne. Capture à faire : onglet **Jobs** → job `Total et moyenne par ville` →
**DAG Visualization**.

## Contrôle : impact des doublons

Le générateur injecte ~2 % de doublons (même `tx_id` et même `status`). Ici on mesure leur poids ;
ils sont supprimés dans la couche Silver (Partie 3).

In [9]:
doublons = (events.map(lambda e: ((e["tx_id"], e["status"]), 1))
                  .reduceByKey(add)
                  .filter(lambda kv: kv[1] > 1)
                  .map(lambda kv: kv[1] - 1)
                  .sum())
print(f"Événements en double : {int(doublons)} ({doublons / n_events:.2%})")

Événements en double : 1028 (2.06%)


## Mesure équitable pour la comparaison avec la Partie 3

Les temps ci-dessus profitent du cache (`completed.cache()`). Pour comparer avec les DataFrames, on mesure
le pipeline complet **depuis le fichier, sans cache** : lecture, parsing, filtre, agrégation par ville.
Médiane de 3 exécutions.

In [10]:
from sahlpay_spark import mediane

def pipeline_rdd_complet():
    return (sc.textFile(HISTORY)
              .map(parse)
              .filter(lambda e: e is not None and e.get("status") == "COMPLETED")
              .map(lambda e: (e["city"], float(e["amount_mad"])))
              .aggregateByKey((0.0, 0),
                              lambda acc, x: (acc[0] + x, acc[1] + 1),
                              lambda a, b: (a[0] + b[0], a[1] + b[1]))
              .mapValues(lambda s: (s[0], s[1], s[0] / s[1]))
              .collect())

mediane(pipeline_rdd_complet, 3, "RDD - pipeline complet depuis le fichier (sans cache)", temps)

[temps] RDD - pipeline complet depuis le fichier (sans cache) : mediane 0.581 s sur 3 executions


0.581

In [11]:
with open(f"{OUT}/temps_partie2.json", "w") as f:
    json.dump(temps, f, indent=2)
pd.Series(temps, name="secondes")

RDD - chargement + count                                 4.328
RDD - filtre COMPLETED + count                           0.655
RDD - total par ville (reduceByKey)                      0.562
RDD - total + moyenne par ville (aggregateByKey)         0.300
RDD - top 5 commerçants (takeOrdered)                    0.328
RDD - pipeline complet depuis le fichier (sans cache)    0.581
Name: secondes, dtype: float64

In [12]:
# Garder la session ouverte le temps de faire les captures de la Spark UI (http://localhost:4040),
# puis exécuter :
# spark.stop()

26/11/25 21:35:14 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 939987 ms exceeds timeout 120000 ms
26/11/25 21:35:14 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 942267 ms exceeds timeout 120000 ms
26/11/25 21:35:14 ERROR TaskSchedulerImpl: Lost executor 1 on 172.28.0.3: Executor heartbeat timed out after 939987 ms
26/11/25 21:35:14 ERROR TaskSchedulerImpl: Lost executor 0 on 172.28.0.6: Executor heartbeat timed out after 942267 ms
26/11/25 21:35:14 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_4_1 !
26/11/25 21:35:14 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_4_0 !
